[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/information_theory/04_kl_divergence_and_f_divergences/first_principles.ipynb)

# Topic 04: KL Divergence and f-Divergences

## 1. First-Principles Intuition & Motivation

Suppose two hypotheses about the world: data comes from $P$, or data comes from $Q$.
After observing $x$, the rational update is governed by the **likelihood ratio** $p(x)/q(x)$ — Bayes' rule multiplies prior odds by exactly this factor.
Its logarithm, $\log\frac{p(x)}{q(x)}$, is the *evidence* the observation provides for $P$ over $Q$, measured in additive units (bits or nats).

The **Kullback–Leibler divergence** is the expected evidence per observation when $P$ is actually true:

$$
D_{\mathrm{KL}}(P \parallel Q) = \mathbb{E}_{X \sim P}\left[\log\frac{p(X)}{q(X)}\right]
$$

Three consequences follow immediately from this reading:

- It should be nonnegative — on average, true hypotheses accumulate evidence in their own favor (proved rigorously below).
- It should be asymmetric — evidence for $P$ against $Q$ under $P$'s data is a different experiment than the reverse.
- It should shrink under any processing of $x$ — degrading the data can only blur the ability to tell the hypotheses apart.

### The Coding and Learning Readings

- **Coding** (Topic 03): $D_{\mathrm{KL}}(P \parallel Q) = H(P, Q) - H(P)$ is the excess bits per symbol paid for compressing $P$-data with a $Q$-code.
- **Learning**: maximum likelihood minimizes $D_{\mathrm{KL}}(\hat{p}_{\text{data}} \parallel q_\theta)$; variational inference minimizes $D_{\mathrm{KL}}(q_\phi \parallel p_{\text{posterior}})$; RLHF penalizes $D_{\mathrm{KL}}(\pi \parallel \pi_{\text{ref}})$.
- **Testing**: by Stein's lemma, the best error exponent for distinguishing $P$ from $Q$ with $n$ samples is $e^{-n D_{\mathrm{KL}}(P \parallel Q)}$ — KL *is* statistical distinguishability.

One quantity, three operational meanings: this convergence is why KL, and not some other formula, sits at the center of probabilistic machine learning.

### Beyond KL: One Convex Function per Divergence

Writing the divergence as an expectation over $Q$ of a convex function of the ratio $t = p/q$ reveals a template:

$$
D_f(P \parallel Q) = \sum_x q(x)\, f\!\left(\frac{p(x)}{q(x)}\right), \qquad f \text{ convex}, \; f(1) = 0
$$

Different convex $f$ give total variation, $\chi^2$, Hellinger, Jensen–Shannon — each with the same skeleton of guarantees (nonnegativity, data processing) but different tail sensitivities and boundedness.
Choosing a divergence is choosing $f$: how harshly to punish ratio deviations of each kind.

## 2. Rigorous Mathematical Definitions & Theorem Statements

### Definition 2.1 (KL Divergence / Relative Entropy)

For distributions $P, Q$ on alphabet $\mathcal{X}$ with $P \ll Q$ (absolute continuity: $q(x) = 0 \Rightarrow p(x) = 0$),

$$
D_{\mathrm{KL}}(P \parallel Q) = \sum_{x \in \mathcal{X}} p(x)\log\frac{p(x)}{q(x)}
$$

with conventions $0\log\frac{0}{q} = 0$ and $p\log\frac{p}{0} = +\infty$.
For densities, replace the sum by an integral.
$D_{\mathrm{KL}}$ is measured in nats (natural log) or bits (log base 2).

### Definition 2.2 (f-Divergence)

Let $f: (0, \infty) \to \mathbb{R}$ be convex with $f(1) = 0$. The **f-divergence** is

$$
D_f(P \parallel Q) = \sum_x q(x)\, f\!\left(\frac{p(x)}{q(x)}\right)
$$

(with standard limiting conventions at $q(x) = 0$). Principal members:

| Divergence | Generator $f(t)$ | Notes |
|---|---|---|
| KL | $t\ln t$ | asymmetric, unbounded |
| Reverse KL | $-\ln t$ | asymmetric, unbounded |
| Total variation | $\tfrac{1}{2}\vert t - 1\vert$ | metric, bounded by 1 |
| Pearson $\chi^2$ | $(t - 1)^2$ | dominates KL |
| Squared Hellinger | $(\sqrt{t} - 1)^2$ | metric after square root |
| Jensen–Shannon | $t\ln t - (t+1)\ln\tfrac{t+1}{2}$ | symmetric, bounded by $\ln 2$ |

### Definition 2.3 (Jensen–Shannon Divergence)

With mixture $M = \tfrac{1}{2}(P + Q)$,

$$
\mathrm{JS}(P, Q) = \tfrac{1}{2}D_{\mathrm{KL}}(P \parallel M) + \tfrac{1}{2}D_{\mathrm{KL}}(Q \parallel M)
$$

$\mathrm{JS}$ is symmetric, always finite, bounded by $\ln 2$, and $\sqrt{\mathrm{JS}}$ is a metric.

### Theorem Statements

- **Theorem A (Nonnegativity)**: $D_f(P \parallel Q) \ge 0$ for every f-divergence, with equality iff $P = Q$ when $f$ is strictly convex at 1. In particular $D_{\mathrm{KL}} \ge 0$ (Gibbs).
- **Theorem B (Chain rule for KL)**: $D_{\mathrm{KL}}(P_{XY} \parallel Q_{XY}) = D_{\mathrm{KL}}(P_X \parallel Q_X) + \mathbb{E}_{x \sim P_X}\left[D_{\mathrm{KL}}(P_{Y \mid X{=}x} \parallel Q_{Y \mid X{=}x})\right]$.
- **Theorem C (Additivity)**: for product measures, $D_{\mathrm{KL}}(P_1 \otimes P_2 \parallel Q_1 \otimes Q_2) = D_{\mathrm{KL}}(P_1 \parallel Q_1) + D_{\mathrm{KL}}(P_2 \parallel Q_2)$; hence $D_{\mathrm{KL}}(P^{\otimes n} \parallel Q^{\otimes n}) = n D_{\mathrm{KL}}(P \parallel Q)$.
- **Theorem D (Data-processing inequality)**: for any Markov kernel (channel) $K$ mapping $X$ to $Y$: $D_f(KP \parallel KQ) \le D_f(P \parallel Q)$, with equality iff the channel output is a sufficient statistic for distinguishing $P$ from $Q$.
- **Theorem E (Gaussian KL)**: $D_{\mathrm{KL}}\big(\mathcal{N}(\mu_1, \sigma_1^2) \parallel \mathcal{N}(\mu_2, \sigma_2^2)\big) = \ln\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1 - \mu_2)^2}{2\sigma_2^2} - \frac{1}{2}$.
- **Theorem F (Pinsker's inequality)**: $\mathrm{TV}(P, Q) \le \sqrt{\tfrac{1}{2}D_{\mathrm{KL}}(P \parallel Q)}$ (nats).

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 3.1 (Nonnegativity of KL via Jensen's inequality)

**Claim.** $D_{\mathrm{KL}}(P \parallel Q) \ge 0$, with equality iff $P = Q$.

**Step 1.** Write the negative divergence as an expectation of a concave function:

$$
-D_{\mathrm{KL}}(P \parallel Q) = \sum_x p(x)\ln\frac{q(x)}{p(x)} = \mathbb{E}_{P}\left[\ln\frac{q(X)}{p(X)}\right]
$$

**Step 2.** Apply Jensen's inequality to the strictly concave $\ln$:

$$
\mathbb{E}_{P}\left[\ln\frac{q(X)}{p(X)}\right] \le \ln \mathbb{E}_{P}\left[\frac{q(X)}{p(X)}\right] = \ln\sum_{x: p(x) \gt 0} p(x)\frac{q(x)}{p(x)} = \ln\sum_{x: p(x) \gt 0} q(x) \le \ln 1 = 0
$$

**Step 3.** Hence $D_{\mathrm{KL}} \ge 0$. Equality in Jensen for a strictly concave function requires the ratio $q(X)/p(X)$ to be $P$-a.s. constant; combined with equality in the final step ($Q$ concentrated on the support of $P$), the constant is 1, i.e., $P = Q$.

$$
\boxed{D_{\mathrm{KL}}(P \parallel Q) \ge 0, \quad \text{equality} \iff P = Q}
$$

The same two steps prove Theorem A for any f-divergence: $D_f = \mathbb{E}_Q\left[f(p/q)\right] \ge f\big(\mathbb{E}_Q[p/q]\big) = f(1) = 0$. $\blacksquare$

### Proof 3.2 (Chain rule and additivity of KL)

**Claim.** $D_{\mathrm{KL}}(P_{XY} \parallel Q_{XY}) = D_{\mathrm{KL}}(P_X \parallel Q_X) + \mathbb{E}_{P_X}\left[D_{\mathrm{KL}}(P_{Y \mid X} \parallel Q_{Y \mid X})\right]$.

**Step 1.** Factor both joints: $p(x, y) = p(x)p(y \mid x)$ and $q(x, y) = q(x)q(y \mid x)$, so the log-ratio splits:

$$
\log\frac{p(x, y)}{q(x, y)} = \log\frac{p(x)}{q(x)} + \log\frac{p(y \mid x)}{q(y \mid x)}
$$

**Step 2.** Take expectations under $P_{XY}$. The first term depends only on $x$ and gives $D_{\mathrm{KL}}(P_X \parallel Q_X)$.
The second, by the tower property, is

$$
\mathbb{E}_{x \sim P_X}\left[\sum_y p(y \mid x)\log\frac{p(y \mid x)}{q(y \mid x)}\right] = \mathbb{E}_{P_X}\left[D_{\mathrm{KL}}(P_{Y \mid X} \parallel Q_{Y \mid X})\right]
$$

**Step 3 (additivity).** If $P_{XY} = P_1 \otimes P_2$ and $Q_{XY} = Q_1 \otimes Q_2$, the conditional divergences are constant in $x$, giving the sum $D_{\mathrm{KL}}(P_1 \parallel Q_1) + D_{\mathrm{KL}}(P_2 \parallel Q_2)$, and by induction $D_{\mathrm{KL}}(P^{\otimes n} \parallel Q^{\otimes n}) = n D_{\mathrm{KL}}(P \parallel Q)$.

$$
\boxed{\text{KL adds over conditionals, and } n \text{ i.i.d. observations carry } n \text{ times the divergence}}
$$

**Interpretation.** Evidence accumulates linearly with sample size — the seed of Stein's lemma and of why more data separates hypotheses exponentially fast. $\blacksquare$

### Proof 3.3 (Data-processing inequality for f-divergences)

**Claim.** Let a channel $K(y \mid x)$ act on both distributions: $p'(y) = \sum_x K(y \mid x)p(x)$ and $q'(y) = \sum_x K(y \mid x)q(x)$. Then $D_f(P' \parallel Q') \le D_f(P \parallel Q)$.

**Step 1 (the key convexity).** The perspective function $(u, v) \mapsto v f(u/v)$ is *jointly convex* on $u \gt 0, v \gt 0$: for $\lambda_i \ge 0$ summing to 1,

$$
\left(\sum_i \lambda_i v_i\right) f\!\left(\frac{\sum_i \lambda_i u_i}{\sum_i \lambda_i v_i}\right) \le \sum_i \lambda_i\, v_i f\!\left(\frac{u_i}{v_i}\right)
$$

This is Jensen's inequality for $f$ applied with weights $\frac{\lambda_i v_i}{\sum_j \lambda_j v_j}$ at the points $u_i / v_i$.

**Step 2.** Fix $y$ and apply Step 1 with $u_x = K(y \mid x)p(x)$, $v_x = K(y \mid x)q(x)$ (uniform $\lambda$ absorbed):

$$
q'(y) f\!\left(\frac{p'(y)}{q'(y)}\right) \le \sum_x K(y \mid x)\, q(x) f\!\left(\frac{p(x)}{q(x)}\right)
$$

**Step 3.** Sum over $y$; the channel weights integrate out ($\sum_y K(y \mid x) = 1$):

$$
D_f(P' \parallel Q') = \sum_y q'(y) f\!\left(\frac{p'(y)}{q'(y)}\right) \le \sum_x q(x) f\!\left(\frac{p(x)}{q(x)}\right) = D_f(P \parallel Q)
$$

$$
\boxed{D_f(KP \parallel KQ) \le D_f(P \parallel Q) \text{ for every channel } K}
$$

**Interpretation.** Quantization, subsampling, adding noise, taking summaries — every processing step can only make two hypotheses harder to distinguish. Equality holds exactly when the processing is sufficient (loses nothing about the $P$-vs-$Q$ question). $\blacksquare$

### Proof 3.4 (KL between two Gaussians)

**Claim.** For $P = \mathcal{N}(\mu_1, \sigma_1^2)$ and $Q = \mathcal{N}(\mu_2, \sigma_2^2)$:

$$
D_{\mathrm{KL}}(P \parallel Q) = \ln\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1 - \mu_2)^2}{2\sigma_2^2} - \frac{1}{2}
$$

**Step 1.** The log-ratio of densities is

$$
\ln\frac{p(x)}{q(x)} = \ln\frac{\sigma_2}{\sigma_1} - \frac{(x - \mu_1)^2}{2\sigma_1^2} + \frac{(x - \mu_2)^2}{2\sigma_2^2}
$$

**Step 2.** Take $\mathbb{E}_{X \sim P}$ term by term:

- $\mathbb{E}_P\left[(X - \mu_1)^2\right] = \sigma_1^2$, so the middle term contributes $-\tfrac{1}{2}$.
- Expand $(X - \mu_2)^2 = (X - \mu_1)^2 + 2(X - \mu_1)(\mu_1 - \mu_2) + (\mu_1 - \mu_2)^2$; the cross term has zero mean, so $\mathbb{E}_P\left[(X - \mu_2)^2\right] = \sigma_1^2 + (\mu_1 - \mu_2)^2$.

**Step 3.** Assemble:

$$
D_{\mathrm{KL}}(P \parallel Q) = \ln\frac{\sigma_2}{\sigma_1} - \frac{1}{2} + \frac{\sigma_1^2 + (\mu_1 - \mu_2)^2}{2\sigma_2^2}
$$

**Check** at $P = Q$: $\ln 1 + \tfrac{\sigma^2}{2\sigma^2} - \tfrac{1}{2} = 0$. Correct.

$$
\boxed{D_{\mathrm{KL}} = \ln\tfrac{\sigma_2}{\sigma_1} + \tfrac{\sigma_1^2 + (\mu_1 - \mu_2)^2}{2\sigma_2^2} - \tfrac{1}{2}}
$$

The multivariate version, $\tfrac{1}{2}\left[\operatorname{tr}(\Sigma_2^{-1}\Sigma_1) + (\mu_2 - \mu_1)^{\top}\Sigma_2^{-1}(\mu_2 - \mu_1) - d + \ln\tfrac{\det\Sigma_2}{\det\Sigma_1}\right]$, follows by the same two-moment computation and is the exact KL term inside every Gaussian VAE. $\blacksquare$

### Proof 3.5 (Pinsker's inequality, with the quadratic bound for Bernoulli pairs)

**Claim.** $\mathrm{TV}(P, Q) \le \sqrt{\tfrac{1}{2} D_{\mathrm{KL}}(P \parallel Q)}$, where $\mathrm{TV} = \tfrac{1}{2}\sum_x \vert p(x) - q(x) \vert$.

**Step 1 (reduce to two points).** Let $A = \{x : p(x) \ge q(x)\}$, and define the 2-point quantization $X \mapsto \mathbf{1}\{X \in A\}$. Then $\mathrm{TV}(P, Q) = P(A) - Q(A) = \mathrm{TV}(\bar{P}, \bar{Q})$ where $\bar{P} = \mathrm{Ber}(a)$, $\bar{Q} = \mathrm{Ber}(b)$ with $a = P(A), b = Q(A)$.
By the data-processing inequality (Proof 3.3), $D_{\mathrm{KL}}(\bar{P} \parallel \bar{Q}) \le D_{\mathrm{KL}}(P \parallel Q)$.
So it suffices to prove the claim for Bernoulli pairs.

**Step 2 (Bernoulli case).** Define for $a, b \in (0, 1)$:

$$
g(a, b) = a\ln\frac{a}{b} + (1-a)\ln\frac{1-a}{1-b} - 2(a - b)^2
$$

At $b = a$, $g = 0$. Differentiate in $b$:

$$
\frac{\partial g}{\partial b} = -\frac{a}{b} + \frac{1-a}{1-b} + 4(a - b) = \frac{b - a}{b(1 - b)} + 4(a - b) = (a - b)\left(4 - \frac{1}{b(1-b)}\right)
$$

Since $b(1 - b) \le \tfrac{1}{4}$, the bracket is $\le 0$, so $g$ decreases as $b$ moves toward $a$ and increases as $b$ moves away — i.e., $g \ge 0$ everywhere with minimum 0 at $b = a$.
Hence $D_{\mathrm{KL}}(\mathrm{Ber}(a) \parallel \mathrm{Ber}(b)) \ge 2(a - b)^2$.

**Step 3.** Combine: $D_{\mathrm{KL}}(P \parallel Q) \ge D_{\mathrm{KL}}(\bar{P} \parallel \bar{Q}) \ge 2(a - b)^2 = 2\,\mathrm{TV}(P, Q)^2$.

$$
\boxed{\mathrm{TV}(P, Q) \le \sqrt{\tfrac{1}{2}D_{\mathrm{KL}}(P \parallel Q)}}
$$

**Interpretation.** Small KL forces small total variation — every event's probability under $P$ and $Q$ differs by at most $\sqrt{\mathrm{KL}/2}$. The converse fails: TV can be tiny while KL is infinite (support mismatch). $\blacksquare$

### Proof 3.6 (Forward vs reverse KL: moment matching vs mode seeking)

**Setup.** Approximate a bimodal target $p = \tfrac{1}{2}\mathcal{N}(-c, 1) + \tfrac{1}{2}\mathcal{N}(c, 1)$ (large $c$) by a single Gaussian $q = \mathcal{N}(\mu, \sigma^2)$.

**Forward KL.** Minimizing $D_{\mathrm{KL}}(p \parallel q) = \mathbb{E}_p[\ln p] - \mathbb{E}_p[\ln q]$ over $(\mu, \sigma^2)$ maximizes $\mathbb{E}_p[\ln q]$.
Setting gradients to zero:

$$
\frac{\partial}{\partial\mu}\mathbb{E}_p[\ln q] = \frac{\mathbb{E}_p[X] - \mu}{\sigma^2} = 0 \implies \mu^* = \mathbb{E}_p[X] = 0
$$

$$
\frac{\partial}{\partial\sigma^2}\mathbb{E}_p[\ln q] = 0 \implies \sigma^{2*} = \mathrm{Var}_p(X) = 1 + c^2
$$

The optimum **matches the moments** of $p$: a single wide Gaussian straddling both modes, placing large mass where $p$ has almost none (between the modes). Mass-covering.

**Reverse KL.** $D_{\mathrm{KL}}(q \parallel p) = \mathbb{E}_q\left[\ln\frac{q(X)}{p(X)}\right]$ explodes wherever $q$ puts mass but $p \approx 0$.
A straddling $q$ centered at 0 pays $-\ln p(0) \approx \tfrac{c^2}{2}$ inside the mass-less valley, while $q \approx \mathcal{N}(\pm c, 1)$ (one mode) achieves cost $\approx \ln 2$ (only the missing-mode mass is ignored, which reverse KL does not punish).
For large $c$ the optimum therefore **locks onto a single mode**. Mode-seeking, and the objective has two symmetric local minima.

$$
\boxed{\arg\min_q D_{\mathrm{KL}}(p \parallel q): \text{ moment-matching}; \quad \arg\min_q D_{\mathrm{KL}}(q \parallel p): \text{ mode-seeking}}
$$

**Interpretation.** MLE/distillation (forward) produce inclusive, sometimes blurry models; variational inference and RLHF (reverse) produce confident, sometimes mode-collapsed ones. $\blacksquare$

## 4. Computational & Algorithmic Insights

### Computing KL Safely

- **Log-space throughout**: compute $\log p - \log q$ from log-probabilities (e.g., `log_softmax` outputs); never form the ratio $p/q$ of raw probabilities.
- **Support hygiene**: mask or smooth zero cells; one empty $q$-bin with $p$-mass makes KL infinite (and gradients NaN). Additive smoothing trades a small bias for finiteness.
- **Framework convention**: `torch.nn.functional.kl_div(input, target)` expects `input` = *log*-probabilities of $Q$ and `target` = probabilities of $P$, computing $D_{\mathrm{KL}}(P \parallel Q)$ — the argument order reverses the math notation, a classic bug source.
- **Per-token KL in RLHF**: accumulate $\log\pi(a_t \mid s_t) - \log\pi_{\text{ref}}(a_t \mid s_t)$ along sampled trajectories; this is an unbiased single-sample estimate of the (reverse) KL under the sampling policy.

### Estimating KL from Samples

Exact KL needs densities; practice offers samples. Three standard estimators:

1. **Plug-in (histogram/kNN)**: binned or k-nearest-neighbor density estimates; biased and dimension-cursed, but simple in low dimension.
2. **Monte Carlo with known densities**: $\widehat{D} = \frac{1}{N}\sum_i \left[\log p(x_i) - \log q(x_i)\right]$ with $x_i \sim p$; unbiased, variance can be reduced by control variates. A popular low-variance surrogate for $x_i \sim q$ is $\frac{1}{N}\sum_i \left(r_i - 1 - \ln r_i\right)$ with $r_i = p(x_i)/q(x_i)$, which is nonnegative term-by-term.
3. **Variational (Donsker–Varadhan)**: $D_{\mathrm{KL}}(P \parallel Q) = \sup_T \left\{\mathbb{E}_P[T(X)] - \ln\mathbb{E}_Q\left[e^{T(X)}\right]\right\}$ over test functions $T$; parameterizing $T$ by a neural network gives the MINE estimator — a lower bound that tightens with optimization.

## 5. Real-World Physics & AI/ML Applications

### Variational Autoencoders and Diffusion

The Gaussian closed form (Proof 3.4) is executed literally in every VAE forward pass: the regularizer $D_{\mathrm{KL}}\big(q_\phi(z \mid x) \parallel \mathcal{N}(0, I)\big)$ has the exact expression $\tfrac{1}{2}\sum_j\left(\mu_j^2 + \sigma_j^2 - 1 - \ln\sigma_j^2\right)$, differentiable in the encoder outputs.
Diffusion models train on a sum of per-timestep KL terms between Gaussian posteriors — the ELBO of Topic 06 unrolled along the noising chain.

### RLHF, Trust Regions, and Distillation

- **RLHF** maximizes reward minus $\beta D_{\mathrm{KL}}(\pi_\theta \parallel \pi_{\text{ref}})$ — a reverse-KL leash keeping the tuned model within the reference model's support, preventing reward hacking into degenerate text. The optimal solution is the tilted distribution $\pi^*(y) \propto \pi_{\text{ref}}(y)e^{r(y)/\beta}$.
- **TRPO/PPO** constrain each policy update by $D_{\mathrm{KL}}(\pi_{\text{old}} \parallel \pi_{\text{new}}) \le \delta$; the KL's local quadratic form $\tfrac{1}{2}\Delta\theta^{\top}F\Delta\theta$ (Fisher information $F$) makes natural gradient the geometry-correct step.
- **Distillation** minimizes forward KL from teacher to student — mass-covering, so the student inherits the teacher's full distribution, not just its argmax.
- **GANs**: the original discriminator objective equals $2\,\mathrm{JS}(p_{\text{data}}, p_G) - \ln 4$ at the optimal discriminator; f-GANs generalize to any f-divergence via the variational representation.

### Statistics and Physics

- **Hypothesis testing**: Stein's lemma — with false-alarm rate held fixed, the miss probability of the optimal test decays as $e^{-n D_{\mathrm{KL}}(P \parallel Q)}$; Sanov's theorem makes KL the rate function of large deviations of empirical distributions.
- **Thermodynamics**: the nonequilibrium free-energy excess of a state $p$ over equilibrium $\pi$ is $F(p) - F(\pi) = k_B T\, D_{\mathrm{KL}}(p \parallel \pi)$ — relaxation to equilibrium is KL descent, and Landauer's bound is its bookkeeping.
- **Bayesian updating**: the expected KL from posterior to prior is the mutual information between parameters and data — the "information gained" by the experiment (Lindley information), used in Bayesian optimal experiment design.

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source |
|---|---|
| Relative entropy, sufficiency | Kullback & Leibler (1951) |
| KL properties, Stein, Sanov | Cover & Thomas (2006), Chapters 2, 11, 12 |
| f-divergences | Csiszár (1967); Polyanskiy & Wu (2024), Part II |
| Pinsker's inequality | Pinsker (1964); Csiszár & Körner (2011) |
| Information geometry, Fisher metric | Amari (2016), *Information Geometry and Its Applications* |
| Gaussian KL in VAEs | Kingma & Welling (2014), *Auto-Encoding Variational Bayes*, Appendix B |
| JS divergence in GANs | Goodfellow et al. (2014); Nowozin et al. (2016), *f-GAN* |
| KL-regularized RL / RLHF | Schulman et al. (2015), *TRPO*; Ouyang et al. (2022), *InstructGPT* |
| Donsker–Varadhan / MINE estimation | Donsker & Varadhan (1975); Belghazi et al. (2018), *MINE* |